# Initialization and Dataset

In [1]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

if Path("open_ragbench").exists():
    print("All set!")
else:
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="vectara/open_ragbench",
        repo_type="dataset",
        local_dir="./open_ragbench"
    )
    print("Dataset ready!")

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

All set!


In [2]:
# Automatically finds all .json files in your directory
corpus_folder = Path("open_ragbench/pdf/arxiv/corpus")
all_json_files = list(corpus_folder.glob("*.json"))
# all_json_files = all_json_files[:50]

print(f"Found {len(all_json_files)} papers ready for indexing!")

Found 1000 papers ready for indexing!


# Indexing

In [3]:
from haystack import Document, Pipeline
from haystack.components.writers import DocumentWriter

In [4]:
from docling.chunking import HybridChunker
from haystack_integrations.components.converters.docling import DoclingConverter, ExportType
from haystack_integrations.document_stores.qdrant import QdrantDocumentStore
from haystack_integrations.components.embedders.vllm import VLLMDocumentEmbedder

/Users/giacomo/.venv-vllm-metal/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
EXPORT_TYPE = ExportType.DOC_CHUNKS
# EMBED_MODEL_ID = "google/embeddinggemma-300m"
EMBED_MODEL_ID = "mlx-community/Qwen3-Embedding-0.6B-8bit"
GENERATION_MODEL_ID = "Qwen/Qwen3-0.6B"

In [15]:
sources = []
for file in all_json_files:
    with open(file) as f:
        paper = json.load(f)

    for section in paper["sections"]:
        sources.append(
            Document(
                content=section["text"][:10],
                meta={
                    "title": paper["title"],
                    "authors": paper["authors"],
                    "abstract": paper["abstract"],
                    "paper_id": paper["id"],
                    "section_id": section["section_id"],
                    "categories": paper["categories"],
                    "updated": paper["updated"],
                    "published": paper["published"]
                }
            )
        )

In [10]:
sources[0]

Document(id=127732e78ab9ce7ccb897f77c2c77f115a8e7c649ab3e9547b0f7f4f7ae44911, content: '#### Abstract

We show that interlacing triangular arrays, introduced by Aggarwal-Borodin-Wheeler to...', meta: {'title': 'Interlacing triangles, Schubert puzzles, and graph colorings', 'authors': ['Christian Gaetz', 'Yibo Gao'], 'abstract': 'We show that interlacing triangular arrays, introduced by\nAggarwal-Borodin-Wheeler to study certain probability measures, can be used to\ncompute structure constants for multiplying Schubert classes in the $K$-theory\nof Grassmannians, in the cohomology of their cotangent bundles, and in the\ncohomology of partial flag varieties. Our results are achieved by establishing\na splitting lemma, allowing for interlacing triangular arrays of high rank to\nbe decomposed into arrays of lower rank, and by constructing a bijection\nbetween interlacing triangular arrays of rank 3 with certain proper vertex\ncolorings of the triangular grid graph that factors through gener

In [16]:
indexing_pipeline = Pipeline()

document_store = QdrantDocumentStore(":memory:", embedding_dim=1024)

In [17]:
converter = DoclingConverter(
    export_type=EXPORT_TYPE, 
    # chunker=HybridChunker(tokenizer=EMBED_MODEL_ID)
    )

indexing_embedder = VLLMDocumentEmbedder(model=EMBED_MODEL_ID,
                                         api_base_url="http://localhost:8001/v1")

writer = DocumentWriter(document_store=document_store)

In [18]:
# indexing_pipeline.add_component("converter", converter)
indexing_pipeline.add_component("embedder", indexing_embedder)
# indexing_pipeline.connect("converter", "embedder")
indexing_pipeline.add_component("writer", writer)
indexing_pipeline.connect("embedder", "writer")

🚅 Components
  - embedder: VLLMDocumentEmbedder
  - writer: DocumentWriter
🛤️ Connections
  - embedder.documents -> writer.documents (list[Document])

In [19]:
indexing_pipeline.run(
    {
        # "converter": {"sources": sources},
        "embedder": {"documents": sources},
    })

Calculating embeddings: 589it [04:14,  2.31it/s]
18900it [00:10, 1869.81it/s]                           


{'embedder': {'meta': {'model': 'mlx-community/Qwen3-Embedding-0.6B-8bit',
   'usage': {'prompt_tokens': 112450,
    'total_tokens': 112450,
    'completion_tokens': 0,
    'prompt_tokens_details': None}}},
 'writer': {'documents_written': 18840}}

In [20]:
docs = document_store.filter_documents()

print(len(docs))

print(docs[0].embedding is not None)

print(type(docs[0].embedding))

print(len(docs[0].embedding) if docs[0].embedding else None)

print(docs[0].embedding[:10] if docs[0].embedding else None)

18840
True
<class 'list'>
1024
[-0.01745232194662094, -0.04936563968658447, -0.011493059806525707, -0.08012977987527847, 0.06204621493816376, -0.012705894187092781, -0.031740762293338776, -0.03070349059998989, -0.004207605961710215, 0.03221259266138077]


In [42]:
from haystack.components.builders.chat_prompt_builder import ChatPromptBuilder
from haystack.dataclasses import ChatMessage
from haystack.components.builders import AnswerBuilder

In [43]:
from haystack_integrations.components.retrievers.qdrant import QdrantEmbeddingRetriever

In [44]:
from haystack_integrations.components.embedders.vllm import VLLMTextEmbedder
from haystack_integrations.components.generators.vllm import VLLMChatGenerator

In [41]:
with open("open_ragbench/pdf/arxiv/queries.json") as f:
    queries = json.load(f)

querys = []
for query in queries:
    querys.append(queries[query]["query"])

querys

['What are the challenges in estimating output impedance in inverter-based grids?',
 'How do changes in effective microbial death rate influence parameters like alpha and beta?',
 'In what areas do syllabic embeddings show potential for improvement based on current research findings?',
 'How does the MLMM approach affect the analysis of Root Mean Squared Error (RMSE)?',
 'How does uncertainty in data affect standard quadratic optimization problems?',
 'What is the concept of location-class independence in object detection?',
 'What happens to coma lobes as beams move further from the feed array center?',
 'How does incorporating demographic factors influence job transition predictions using LLMs?',
 'What challenges arise when using Langevin samplers with complex paths?',
 'How do traditional feature selection methods for classification work?',
 'How are expectations calculated in the narrative framework?',
 'Why is post-calibrated softmax adjustment (PCSA) important in handling class 

In [45]:
chat_template = [
    ChatMessage.from_system(
        """
        You are a helpful AI assistant.
        Use the provided documents to answer the question.
        If the answer is not in the documents, say so.
        """
    ),
    ChatMessage.from_user(
        """
        Documents:
        {% for doc in documents %}
            {{ doc.content }}
        {% endfor %}

        Question:
        {{query}}
        """
    )
]

rag_pipeline = Pipeline()

In [46]:
rag_embedder = VLLMTextEmbedder(model=EMBED_MODEL_ID,
                                api_base_url="http://localhost:8001/v1")

retriever = QdrantEmbeddingRetriever(document_store=document_store, top_k=5)

prompt_builder = ChatPromptBuilder(template=chat_template, required_variables={"query", "documents"})

llm = VLLMChatGenerator(model=GENERATION_MODEL_ID,
                        api_base_url="http://localhost:8000/v1")

answer_builder = AnswerBuilder()

In [47]:
rag_pipeline.add_component("embedder", rag_embedder)
rag_pipeline.add_component("retriever", retriever)
rag_pipeline.connect("embedder.embedding", "retriever")
rag_pipeline.add_component("prompt_builder", prompt_builder)
rag_pipeline.connect("retriever", "prompt_builder")
rag_pipeline.add_component("llm", llm)
rag_pipeline.connect("prompt_builder", "llm")
rag_pipeline.add_component("answer_builder", answer_builder)
rag_pipeline.connect("llm", "answer_builder")
rag_pipeline.connect("retriever", "answer_builder")

🚅 Components
  - embedder: VLLMTextEmbedder
  - retriever: QdrantEmbeddingRetriever
  - prompt_builder: ChatPromptBuilder
  - llm: VLLMChatGenerator
  - answer_builder: AnswerBuilder
🛤️ Connections
  - embedder.embedding -> retriever.query_embedding (list[float])
  - retriever.documents -> prompt_builder.documents (list[Document])
  - retriever.documents -> answer_builder.documents (list[Document])
  - prompt_builder.prompt -> llm.messages (list[ChatMessage])
  - llm.replies -> answer_builder.replies (list[ChatMessage])

In [ ]:
rag_pipeline.show()

In [49]:
response = rag_pipeline.run(
    {
        "embedder": {"text": querys[0]},
        "prompt_builder": {"query": querys[0]},
        "answer_builder": {"query": querys[0]},
    })

In [50]:
response["answer_builder"]["answers"][0].data

'<think>\nOkay, let\'s see. The user is asking about the challenges in estimating output impedance in inverter-based grids. I need to check the provided documents to find relevant information.\n\nLooking at the documents, there\'s a mention of "Estimand" and "Inversio" in the first line. Then there\'s a section with "Varianc" and "Estimat". The question is about output impedance estimation in inverter grids. \n\nI don\'t see any specific information about challenges in estimating output impedance. The documents seem to be about different terms like "Estimat" and "Inversio", but none directly address the challenges. Maybe there\'s a mix-up in the document titles, but based on the given data, I can\'t find any challenges related to output impedance estimation in inverter grids. So the answer would be that the documents don\'t provide information on this topic.\n</think>\n\nThe documents do not provide information about the challenges in estimating output impedance in inverter-based grids